In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════
# EXPLORACIÓN — Estructura ficheros 2010-2018
# ══════════════════════════════════════════════════════════

import pandas as pd
import os

BASE = '/content/drive/MyDrive/TFM_Seguridad_Vial'
carpeta = f'{BASE}/datos/accidentes_general/raw/'

# Exploramos 2010 y 2018 para ver si la estructura es consistente
for año in [2010, 2014, 2018]:
    fichero = f'{carpeta}Accidentes{año}.csv'
    print(f'\n{"="*50}')
    print(f'FICHERO {año}')
    print(f'{"="*50}')
    for enc in ['latin-1', 'utf-8', 'utf-8-sig', 'cp1252']:
        try:
            df = pd.read_csv(fichero, sep=';', encoding=enc, nrows=3)
            print(f'✅ Encoding: {enc}')
            print(f'Columnas: {df.columns.tolist()}')
            print(df.head(2).to_string())
            break
        except Exception as e:
            print(f'❌ {enc}: {e}')


FICHERO 2010
✅ Encoding: latin-1
Columnas: ['FECHA', 'RANGO HORARIO', 'DIA SEMANA', 'DISTRITO', 'LUGAR ACCIDENTE', 'Nº', 'Nº PARTE', 'CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Seco', 'CPFA Nieve', 'CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo', 'CPSV Seca Y Limpia', 'Nº VICTIMAS *', 'TIPO ACCIDENTE', 'Tipo Vehiculo', 'TIPO PERSONA', 'SEXO', 'LESIVIDAD', 'Tramo Edad']
        FECHA     RANGO HORARIO DIA SEMANA                        DISTRITO                                                                                       LUGAR ACCIDENTE   Nº  Nº PARTE CPFA Granizo CPFA Hielo CPFA Lluvia CPFA Niebla CPFA Seco CPFA Nieve CPSV Mojada CPSV Aceite CPSV Barro CPSV Grava Suelta CPSV Hielo CPSV Seca Y Limpia  Nº VICTIMAS *                            TIPO ACCIDENTE                             Tipo Vehiculo TIPO PERSONA    SEXO                                 LESIVIDAD       Tramo Edad
0  01/01/2010  DE 00:00 A 00:59    VIERNES  CHAMARTI

In [3]:
# ══════════════════════════════════════════════════════════
# NOTEBOOK: 01b_ETL_historico_2010_2018.ipynb
# ══════════════════════════════════════════════════════════

# CELDA 1 — Librerías y configuración
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/TFM_Seguridad_Vial'
outputs = f'{BASE}/outputs'

# Lista de ficheros 2010-2018
ficheros_hist = [f'Accidentes{año}.csv' for año in range(2010, 2019)]
carpeta = f'{BASE}/datos/accidentes_general/raw/'

print(f'Ficheros a procesar: {len(ficheros_hist)}')
for f in ficheros_hist:
    existe = os.path.exists(f'{carpeta}{f}')
    print(f'  {f}: {"✅" if existe else "❌"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ficheros a procesar: 9
  Accidentes2010.csv: ✅
  Accidentes2011.csv: ✅
  Accidentes2012.csv: ✅
  Accidentes2013.csv: ✅
  Accidentes2014.csv: ✅
  Accidentes2015.csv: ✅
  Accidentes2016.csv: ✅
  Accidentes2017.csv: ✅
  Accidentes2018.csv: ✅


In [4]:
# CELDA 2 — Mapa de lesividad 2010-2018
# Documentado en: Estructura_accidentes_2010-2018.pdf

mapa_lesividad_hist = {
    'IL': 'sin_asistencia',  # Ileso
    'HG': 'grave',           # Herido grave (hospitalización > 24h)
    'HL': 'leve',            # Herido leve (asistencia ≤ 24h)
    'MU': 'fallecido',       # Muerto en 24 horas
}

jerarquia_gravedad = {
    'fallecido':      4,
    'grave':          3,
    'leve':           2,
    'sin_asistencia': 1,
    'desconocido':    0
}
jerarquia_inversa = {v: k for k, v in jerarquia_gravedad.items()}

print('Mapa de lesividad 2010-2018:')
for k, v in mapa_lesividad_hist.items():
    print(f'  {k} → {v}')

Mapa de lesividad 2010-2018:
  IL → sin_asistencia
  HG → grave
  HL → leve
  MU → fallecido


In [5]:
# CELDA 3 — Cargar y procesar ficheros 2010-2018

# Mapa de nombres de distrito históricos → nombres estándar con tildes
mapa_distritos_hist = {
    'CHAMARTIN': 'CHAMARTÍN',
    'CHAMBERI':  'CHAMBERÍ',
    'SAN BLAS':  'SAN BLAS-CANILLEJAS',
    'TETUAN':    'TETUÁN',
    'VICALVARO': 'VICÁLVARO',
}

def cargar_historico(fichero):
    df = pd.read_csv(fichero, sep=';', encoding='latin-1')

    # Normalizar nombres de columnas
    df.columns = df.columns.str.strip()

    # Unificar columna de víctimas (cambia nombre en 2018)
    if '* Nº VICTIMAS' in df.columns:
        df = df.rename(columns={'* Nº VICTIMAS': 'num_victimas'})
    elif 'Nº VICTIMAS *' in df.columns:
        df = df.rename(columns={'Nº VICTIMAS *': 'num_victimas'})

    # Normalizar columnas clave
    df = df.rename(columns={
        'DISTRITO':      'distrito',
        'TIPO PERSONA':  'tipo_persona',
        'Tipo Vehiculo': 'tipo_vehiculo',
        'LESIVIDAD':     'lesividad_raw',
        'Nº PARTE':      'num_expediente',
        'FECHA':         'fecha'
    })

    # Extraer año y mes
    df['fecha_dt'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
    df['año']      = df['fecha_dt'].dt.year
    df['mes']      = df['fecha_dt'].dt.month

    # Normalizar distrito y aplicar mapa de nombres históricos
    df['distrito'] = (
        df['distrito']
        .str.strip()
        .str.upper()
        .str.replace(' - ', '-', regex=False)
    )
    df['distrito'] = df['distrito'].replace(mapa_distritos_hist)

    # Mapear lesividad
    df['lesividad_raw'] = df['lesividad_raw'].str.strip().str.upper()
    df['gravedad'] = df['lesividad_raw'].map(mapa_lesividad_hist).fillna('sin_asistencia')

    return df

# Cargamos todos los ficheros
dfs_hist = []
for f in ficheros_hist:
    ruta = f'{carpeta}{f}'
    try:
        df = cargar_historico(ruta)
        dfs_hist.append(df)
        print(f'✅ {f}: {len(df)} filas')
    except Exception as e:
        print(f'❌ {f}: {e}')

historico = pd.concat(dfs_hist, ignore_index=True)
print(f'\nTotal filas: {len(historico)}')
print(f'Años: {sorted(historico["año"].unique())}')
print(f'\nDistribución gravedad:')
print(historico['gravedad'].value_counts())

✅ Accidentes2010.csv: 26578 filas
✅ Accidentes2011.csv: 27342 filas
✅ Accidentes2012.csv: 26982 filas
✅ Accidentes2013.csv: 26839 filas
✅ Accidentes2014.csv: 27967 filas
✅ Accidentes2015.csv: 28172 filas
✅ Accidentes2016.csv: 29201 filas
✅ Accidentes2017.csv: 29795 filas
✅ Accidentes2018.csv: 30122 filas

Total filas: 252998
Años: [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018)]

Distribución gravedad:
gravedad
sin_asistencia    141432
leve              102716
grave               8850
Name: count, dtype: int64


In [6]:
# CELDA 4 — Filtrar peatones y ciclistas

# Verificamos valores únicos de tipo_persona y tipo_vehiculo
print('Valores tipo_persona:')
print(historico['tipo_persona'].value_counts().head(10))
print('\nValores tipo_vehiculo (bicicleta):')
print(historico[historico['tipo_vehiculo'].str.contains('BICI', na=False, case=False)]['tipo_vehiculo'].value_counts())

Valores tipo_persona:
tipo_persona
CONDUCTOR                                   158734
VIAJERO                                      47480
TESTIGO                                      31666
PEATON                                       15118
Name: count, dtype: int64

Valores tipo_vehiculo (bicicleta):
tipo_vehiculo
BICICLETA                                   5524
Name: count, dtype: int64


In [7]:
# CELDA 5 — Definir función y filtrar peatones y ciclistas

def agregar_historico(df, grupo, filtro_col, filtro_val):
    """
    Filtra por grupo (peatón o ciclista) y agrega por distrito/año/mes.
    Usa gravedad máxima por expediente — mismo criterio que ETL 2019-2024.
    """
    if filtro_col == 'tipo_persona':
        df_grupo = df[df[filtro_col].str.upper().str.strip() == filtro_val.upper()].copy()
    else:
        df_grupo = df[df[filtro_col].str.upper().str.contains(filtro_val.upper(), na=False)].copy()

    print(f'{grupo}: {len(df_grupo)} filas')

    # Gravedad máxima por expediente, distrito, año, mes
    df_grupo['gravedad_num'] = df_grupo['gravedad'].map(jerarquia_gravedad).fillna(0)
    grav_max = (
        df_grupo.groupby(['distrito','año','mes','num_expediente'])['gravedad_num']
        .max().reset_index()
    )
    grav_max['gravedad_final'] = grav_max['gravedad_num'].map(jerarquia_inversa)

    # Agregar por distrito, año, mes
    base     = grav_max.groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_total'})
    leves    = grav_max[grav_max['gravedad_final']=='leve'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_leves'})
    graves   = grav_max[grav_max['gravedad_final']=='grave'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_graves'})
    mortales = grav_max[grav_max['gravedad_final']=='fallecido'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_mortales'})

    r = base.merge(leves,    on=['distrito','año','mes'], how='left')
    r = r.merge(graves,      on=['distrito','año','mes'], how='left')
    r = r.merge(mortales,    on=['distrito','año','mes'], how='left')

    cols = [f'acc_{grupo}_leves', f'acc_{grupo}_graves', f'acc_{grupo}_mortales']
    r[cols] = r[cols].fillna(0).astype(int)

    return r

# Peatones: tipo_persona == 'PEATON'
acc_peatones_hist = agregar_historico(
    historico, 'peatones', 'tipo_persona', 'PEATON'
)

# Ciclistas: tipo_vehiculo == 'BICICLETA'
acc_bici_hist = agregar_historico(
    historico, 'bici', 'tipo_vehiculo', 'BICICLETA'
)

print(f'\nForma peatones histórico: {acc_peatones_hist.shape}')
print(f'Forma bici histórico:     {acc_bici_hist.shape}')
print(f'\nEjemplo peatones:')
print(acc_peatones_hist.head(3).to_string())

peatones: 15118 filas
bici: 5524 filas

Forma peatones histórico: (2215, 7)
Forma bici histórico:     (1817, 7)

Ejemplo peatones:
     distrito   año  mes  acc_peatones_total  acc_peatones_leves  acc_peatones_graves  acc_peatones_mortales
0  ARGANZUELA  2010    1                   2                   1                    1                      0
1  ARGANZUELA  2010    2                   4                   2                    2                      0
2  ARGANZUELA  2010    3                  10                   9                    0                      0


In [8]:
# CELDA 5B — VRU histórico sin doble conteo

orden_gravedad  = {'desconocido': 0, 'sin_asistencia': 1, 'leve': 2, 'grave': 3, 'fallecido': 4}
rank_a_gravedad = {v: k for k, v in orden_gravedad.items()}

# Re-filtramos bici y peatones desde historico para deduplicar
bici_hist_raw = historico[
    historico['tipo_vehiculo'].str.upper().str.contains('BICI', na=False)
].copy()

peat_hist_raw = historico[
    historico['tipo_persona'].str.upper().str.strip() == 'PEATON'
].copy()

# Unimos y deduplicamos por expediente
vru_hist = pd.concat([
    bici_hist_raw.assign(origen='bici'),
    peat_hist_raw.assign(origen='peaton')
], ignore_index=True)

vru_hist['gravedad_rank'] = vru_hist['gravedad'].map(orden_gravedad)

vru_unicos_hist = (
    vru_hist.groupby(['distrito', 'año', 'mes', 'num_expediente'], as_index=False)
            .agg(gravedad_rank=('gravedad_rank', 'max'))
)
vru_unicos_hist['gravedad_max'] = vru_unicos_hist['gravedad_rank'].map(rank_a_gravedad)

acc_vru_hist = (
    vru_unicos_hist
    .groupby(['distrito', 'año', 'mes'])
    .agg(
        acc_vru_total   =('num_expediente', 'nunique'),
        acc_vru_leves   =('gravedad_max', lambda x: (x == 'leve').sum()),
        acc_vru_graves  =('gravedad_max', lambda x: (x == 'grave').sum()),
        acc_vru_mortales=('gravedad_max', lambda x: (x == 'fallecido').sum())
    )
    .reset_index()
)
acc_vru_hist['acc_ponderado'] = (
    acc_vru_hist['acc_vru_leves']    * 1 +
    acc_vru_hist['acc_vru_graves']   * 7 +
    acc_vru_hist['acc_vru_mortales'] * 16
)

# Diagnóstico
exp_bici_h = set(bici_hist_raw['num_expediente'].dropna().unique())
exp_peat_h = set(peat_hist_raw['num_expediente'].dropna().unique())
solapados_h = exp_bici_h.intersection(exp_peat_h)
print(f"✅ Expedientes solapados histórico (2010-2018): {len(solapados_h)}")
print(f"Forma acc_vru_hist: {acc_vru_hist.shape}")

✅ Expedientes solapados histórico (2010-2018): 572
Forma acc_vru_hist: (2245, 8)


In [9]:
# CELDA 6 — Construir tabla mensual histórica completa

import itertools

# Lista fija de 21 distritos
distritos_lista = sorted([
    'ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTÍN',
    'CHAMBERÍ', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA',
    'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS',
    'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUÁN',
    'USERA', 'VICÁLVARO', 'VILLA DE VALLECAS', 'VILLAVERDE'
])
años_hist   = list(range(2010, 2019))
meses_lista = list(range(1, 13))

# Grid completo 21 × 9 × 12 = 2268 combinaciones
tabla_hist = pd.DataFrame(
    list(itertools.product(distritos_lista, años_hist, meses_lista)),
    columns=['distrito', 'año', 'mes']
)
print(f'Combinaciones totales: {len(tabla_hist)}')  # debe ser 2268

# Unimos bici y peatones por separado
tabla_hist = tabla_hist.merge(acc_bici_hist,     on=['distrito', 'año', 'mes'], how='left')
tabla_hist = tabla_hist.merge(acc_peatones_hist, on=['distrito', 'año', 'mes'], how='left')

# Rellenamos nulos con 0
cols_acc = [
    'acc_bici_total',     'acc_bici_leves',     'acc_bici_graves',     'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves',  'acc_peatones_graves', 'acc_peatones_mortales'
]
tabla_hist[cols_acc] = tabla_hist[cols_acc].fillna(0).astype(int)

# ── VRU sin doble conteo — desde acc_vru_hist ────────────────────────────
tabla_hist = tabla_hist.merge(
    acc_vru_hist[[
        'distrito', 'año', 'mes',
        'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
        'acc_vru_mortales', 'acc_ponderado'
    ]],
    on=['distrito', 'año', 'mes'], how='left'
)
cols_vru = ['acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
            'acc_vru_mortales', 'acc_ponderado']
tabla_hist[cols_vru] = tabla_hist[cols_vru].fillna(0)

# Verificación
assert (tabla_hist['acc_vru_leves'] + tabla_hist['acc_vru_graves'] +
        tabla_hist['acc_vru_mortales'] <= tabla_hist['acc_vru_total']).all(), \
    "⚠️ Inconsistencia en VRU histórico"
print("✅ VRU histórico sin doble conteo")

# Resumen
print(f'\nForma: {tabla_hist.shape}')
print(f'Nulos: {tabla_hist.isnull().sum().sum()}')
print(f'\nAcc_ponderado por año:')
print(tabla_hist.groupby('año')['acc_ponderado'].sum().to_string())

Combinaciones totales: 2268
✅ VRU histórico sin doble conteo

Forma: (2268, 16)
Nulos: 0

Acc_ponderado por año:
año
2010    4102.0
2011    4464.0
2012    4159.0
2013    4413.0
2014    4575.0
2015    4149.0
2016    4496.0
2017    4216.0
2018    4181.0


In [10]:
# CELDA 7 — Tabla anual histórica (agregación de mensual)

tabla_anual_hist = (
    tabla_hist.groupby(['distrito','año'])
    .agg(
        acc_bici_total     = ('acc_bici_total',     'sum'),
        acc_bici_leves     = ('acc_bici_leves',     'sum'),
        acc_bici_graves    = ('acc_bici_graves',    'sum'),
        acc_bici_mortales  = ('acc_bici_mortales',  'sum'),
        acc_peatones_total = ('acc_peatones_total', 'sum'),
        acc_peatones_leves = ('acc_peatones_leves', 'sum'),
        acc_peatones_graves= ('acc_peatones_graves','sum'),
        acc_peatones_mortales=('acc_peatones_mortales','sum'),
        acc_vru_leves      = ('acc_vru_leves',      'sum'),
        acc_vru_graves     = ('acc_vru_graves',     'sum'),
        acc_vru_mortales   = ('acc_vru_mortales',   'sum'),
        acc_vru_total      = ('acc_vru_total',      'sum'),
        acc_ponderado      = ('acc_ponderado',      'sum'),
    )
    .reset_index()
)

print(f'Forma tabla anual histórica: {tabla_anual_hist.shape}')
print(f'Años: {sorted(tabla_anual_hist["año"].unique())}')
print(f'Distritos: {tabla_anual_hist["distrito"].nunique()}')

# Vista del acc_ponderado por distrito y año
pivot = tabla_anual_hist.pivot_table(
    values='acc_ponderado',
    index='distrito',
    columns='año',
    aggfunc='sum'
)
print('\n=== ACC_PONDERADO ANUAL — 2010-2018 ===')
print(pivot.to_string())

Forma tabla anual histórica: (189, 15)
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Distritos: 21

=== ACC_PONDERADO ANUAL — 2010-2018 ===
año                   2010   2011   2012   2013   2014   2015   2016   2017   2018
distrito                                                                          
ARGANZUELA           175.0  158.0  150.0  171.0  214.0  213.0  137.0  187.0  157.0
BARAJAS               61.0   72.0   66.0   58.0   62.0   68.0  136.0   62.0  108.0
CARABANCHEL          288.0  296.0  292.0  318.0  344.0  278.0  292.0  337.0  237.0
CENTRO               424.0  341.0  309.0  323.0  396.0  334.0  435.0  472.0  436.0
CHAMARTÍN            222.0  207.0  250.0  232.0  245.0  241.0  210.0  178.0  179.0
CHAMBERÍ             261.0  269.0  250.0  283.0  265.0  197.0  214.0  218.0  233.0
CIUDAD LINEAL        238.0  308.0  200.0  228.0  233.0  311.0  231.0  200.0  249.0
FUENCARR

In [11]:
# CELDA 8 — Unificar con tabla 2019-2024 y guardar

# Cargamos tabla anual 2019-2024
tabla_reciente = pd.read_csv(f'{outputs}/tabla_anual_final.csv')

# Columnas comunes para la unificación
cols_comunes = [
    'distrito', 'año',
    'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales',
    'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_vru_total',
    'acc_ponderado'
]

# Unificamos — solo columnas comunes para la serie histórica
tabla_serie_completa = pd.concat([
    tabla_anual_hist[cols_comunes],
    tabla_reciente[cols_comunes]
], ignore_index=True).sort_values(['distrito','año']).reset_index(drop=True)

print(f'Forma serie completa: {tabla_serie_completa.shape}')
print(f'Años: {sorted(tabla_serie_completa["año"].unique())}')
print(f'Distritos: {tabla_serie_completa["distrito"].nunique()}')

# Verificación — todos los distritos tienen 16 años (2010-2025)
conteo = tabla_serie_completa.groupby('distrito')['año'].count()
print(f'\nAños por distrito:')
print(conteo.value_counts().to_string())

# Guardamos
tabla_serie_completa.to_csv(
    f'{outputs}/serie_historica_2010_2024.csv',
    index=False, encoding='utf-8-sig'
)
print('\n✅ serie_historica_2010_2024.csv guardada')

# Vista final acc_ponderado por año
print('\n=== ACC_PONDERADO MEDIO POR AÑO (todos los distritos) ===')
print(
    tabla_serie_completa.groupby('año')['acc_ponderado']
    .mean().round(1).to_string()
)

Forma serie completa: (315, 15)
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Distritos: 21

Años por distrito:
año
15    21

✅ serie_historica_2010_2024.csv guardada

=== ACC_PONDERADO MEDIO POR AÑO (todos los distritos) ===
año
2010    195.3
2011    212.6
2012    198.0
2013    210.1
2014    217.9
2015    197.6
2016    214.1
2017    200.8
2018    199.1
2019    170.6
2020    133.9
2021    138.8
2022    146.5
2023    156.8
2024    159.3


In [12]:
# CELDA 9 — Tratamiento de distorsiones en la serie histórica
# Dos tratamientos documentados:
# 1. COVID 2020: interpolación lineal selectiva (solo donde hubo caída real)
# 2. Cambio estructural 2019: dummy binaria (cambio de sistema de registro PEA→SIGIT)
# Referencia: Nardo et al. (2008), tratamiento de outliers en series temporales

serie = pd.read_csv(f'{outputs}/serie_historica_2010_2024.csv')
serie_corregida = serie.copy()

# ── Tratamiento 1: COVID 2020 — interpolación lineal selectiva ────────────
# Solo se interpola en distritos donde 2020 fue al menos 10% inferior
# a la media de 2019 y 2021. En distritos sin caída significativa
# se mantiene el valor real para no introducir distorsiones artificiales.
# Justificación: caída heterogénea entre distritos (desde -54% en Chamartín
# hasta +45% en Vicálvaro), por lo que una corrección universal sería
# metodológicamente incorrecta.

distritos_interpolados = []
distritos_sin_cambio   = []

for distrito in serie['distrito'].unique():
    mask_2020 = (serie_corregida['distrito'] == distrito) & (serie_corregida['año'] == 2020)
    val_2019  = serie[(serie['distrito'] == distrito) & (serie['año'] == 2019)]['acc_ponderado'].values
    val_2021  = serie[(serie['distrito'] == distrito) & (serie['año'] == 2021)]['acc_ponderado'].values
    val_2020  = serie[(serie['distrito'] == distrito) & (serie['año'] == 2020)]['acc_ponderado'].values

    if len(val_2019) > 0 and len(val_2021) > 0 and len(val_2020) > 0:
        interpolado = (val_2019[0] + val_2021[0]) / 2
        if val_2020[0] < interpolado * 0.90:
            serie_corregida.loc[mask_2020, 'acc_ponderado'] = round(interpolado)
            distritos_interpolados.append(distrito)
        else:
            distritos_sin_cambio.append(distrito)

# Columna de control: 1 = valor imputado, 0 = valor real
serie_corregida['covid_imputado'] = 0
for d in distritos_interpolados:
    serie_corregida.loc[
        (serie_corregida['distrito'] == d) & (serie_corregida['año'] == 2020),
        'covid_imputado'
    ] = 1

# ── Tratamiento 2: Cambio estructural 2019 — dummy binaria ───────────────
# 0 = datos aplicación PEA (2010-2018)
# 1 = datos aplicación SIGIT (2019-2024)
# Justificación: cambio de sistema de registro del Ayuntamiento produjo
# discontinuidad estadística entre ambos períodos (-13% en media).
# Se controla mediante variable indicadora en modelos que admiten
# variables exógenas (ARIMAX, regresión de panel).
# En SARIMA puro se documenta como limitación metodológica.

serie_corregida['cambio_estructura'] = (serie_corregida['año'] >= 2019).astype(int)

# ── Verificación ─────────────────────────────────────────────────────────
print(f'Distritos interpolados ({len(distritos_interpolados)}):')
for d in sorted(distritos_interpolados):
    print(f'  {d}')

print(f'\nDistritos sin cambio ({len(distritos_sin_cambio)}):')
for d in sorted(distritos_sin_cambio):
    print(f'  {d}')

print('\n=== COMPARACIÓN 2020: REAL vs IMPUTADO ===')
comp  = serie[serie['año']==2020][['distrito','acc_ponderado']].rename(columns={'acc_ponderado':'real'})
comp2 = serie_corregida[serie_corregida['año']==2020][['distrito','acc_ponderado']].rename(columns={'acc_ponderado':'imputado'})
comp  = comp.merge(comp2, on='distrito')
comp['diferencia'] = comp['imputado'] - comp['real']
print(comp.sort_values('diferencia', ascending=False).to_string(index=False))

print('\n=== ACC_PONDERADO MEDIO POR AÑO (tras corrección selectiva) ===')
print(serie_corregida.groupby('año')['acc_ponderado'].mean().round(1).to_string())

print(f'\nColumnas de control añadidas:')
print(f'  covid_imputado    → {serie_corregida["covid_imputado"].sum()} filas marcadas')
print(f'  cambio_estructura → {serie_corregida["cambio_estructura"].sum()} filas marcadas')

# ── Guardar ──────────────────────────────────────────────────────────────
serie_corregida.to_csv(
    f'{outputs}/serie_historica_2010_2024_corregida.csv',
    index=False, encoding='utf-8-sig'
)
print('\n✅ serie_historica_2010_2024_corregida.csv guardada')
print(f'Forma: {serie_corregida.shape}')
print(f'Columnas: {serie_corregida.columns.tolist()}')

Distritos interpolados (12):
  ARGANZUELA
  BARAJAS
  CARABANCHEL
  CENTRO
  CHAMARTÍN
  CHAMBERÍ
  CIUDAD LINEAL
  FUENCARRAL-EL PARDO
  MORATALAZ
  PUENTE DE VALLECAS
  VILLA DE VALLECAS
  VILLAVERDE

Distritos sin cambio (9):
  HORTALEZA
  LATINA
  MONCLOA-ARAVACA
  RETIRO
  SALAMANCA
  SAN BLAS-CANILLEJAS
  TETUÁN
  USERA
  VICÁLVARO

=== COMPARACIÓN 2020: REAL vs IMPUTADO ===
           distrito  real  imputado  diferencia
             CENTRO 142.0     280.0       138.0
          CHAMARTÍN 120.0     200.0        80.0
         ARGANZUELA 132.0     204.0        72.0
           CHAMBERÍ 129.0     196.0        67.0
 PUENTE DE VALLECAS 171.0     220.0        49.0
      CIUDAD LINEAL 107.0     149.0        42.0
         VILLAVERDE  85.0     122.0        37.0
FUENCARRAL-EL PARDO 175.0     200.0        25.0
        CARABANCHEL 146.0     168.0        22.0
  VILLA DE VALLECAS  88.0     108.0        20.0
            BARAJAS  37.0      56.0        19.0
          MORATALAZ  49.0      60.0     

In [13]:
# CELDA 10 — Exportar tabla mensual histórica 2010-2018

# tabla_hist ya está en memoria desde la celda 6
cols_comunes_mensual = [
    'distrito', 'año', 'mes',
    'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales',
    'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_vru_total',
    'acc_ponderado'
]

tabla_mensual_hist_export = tabla_hist[cols_comunes_mensual].copy()

print(f'Forma: {tabla_mensual_hist_export.shape}')
print(f'Años: {sorted(tabla_mensual_hist_export["año"].unique())}')
print(f'Distritos: {tabla_mensual_hist_export["distrito"].nunique()}')

# Verificación: todos los distritos deben tener 108 meses (9 años × 12 meses)
conteo = tabla_mensual_hist_export.groupby('distrito').size()
print(f'\nObservaciones por distrito:')
print(conteo.value_counts().to_string())

# Media mensual por año
print('\n=== ACC_PONDERADO MEDIO POR AÑO ===')
print(tabla_mensual_hist_export.groupby('año')['acc_ponderado'].mean().round(2).to_string())

# Guardamos
tabla_mensual_hist_export.to_csv(
    f'{outputs}/serie_mensual_2010_2018.csv',
    index=False, encoding='utf-8-sig'
)
print('\n✅ serie_mensual_2010_2018.csv guardada')

Forma: (2268, 16)
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Distritos: 21

Observaciones por distrito:
108    21

=== ACC_PONDERADO MEDIO POR AÑO ===
año
2010    16.28
2011    17.71
2012    16.50
2013    17.51
2014    18.15
2015    16.46
2016    17.84
2017    16.73
2018    16.59

✅ serie_mensual_2010_2018.csv guardada


In [14]:
# CELDA 11 — Unificar serie mensual 2010-2024

tabla_mensual_reciente = pd.read_csv(f'{outputs}/tabla_mensual_final.csv')

cols_comunes_mensual = [
    'distrito', 'año', 'mes',
    'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales',
    'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_vru_total',
    'acc_ponderado'
]

# Unificamos
tabla_mensual_completa = pd.concat([
    tabla_mensual_hist_export[cols_comunes_mensual],
    tabla_mensual_reciente[cols_comunes_mensual]
], ignore_index=True).sort_values(['distrito','año','mes']).reset_index(drop=True)

# Variables de control
tabla_mensual_completa['cambio_estructura'] = (
    tabla_mensual_completa['año'] >= 2019
).astype(int)

# Marcamos período COVID — los 8 meses con caída > 15% respecto a media normal
# Coincide con: Estado de Alarma (marzo-junio 2020, RD 463/2020)
# y restricciones segunda ola (septiembre-noviembre 2020)
# Nota: febrero incluido por caída del 18.6%, próxima al umbral del 15%
tabla_mensual_completa['covid_periodo'] = (
    (tabla_mensual_completa['año'] == 2020) &
    (tabla_mensual_completa['mes'].isin([2, 3, 4, 5, 6, 9, 10, 11]))
).astype(int)

# Verificación
print(f'Forma: {tabla_mensual_completa.shape}')
print(f'Años: {sorted(tabla_mensual_completa["año"].unique())}')
print(f'Distritos: {tabla_mensual_completa["distrito"].nunique()}')

conteo = tabla_mensual_completa.groupby('distrito').size()
print(f'\nObservaciones por distrito: {conteo.unique()[0]} (esperado: 180)')

print(f'\ncovid_periodo marcado: {tabla_mensual_completa["covid_periodo"].sum()} filas')
print(f'(esperado: 8 meses × 21 distritos = 168)')

print('\n=== ACC_PONDERADO MEDIO POR AÑO ===')
print(tabla_mensual_completa.groupby('año')['acc_ponderado'].mean().round(2).to_string())

tabla_mensual_completa.to_csv(
    f'{outputs}/serie_mensual_2010_2024.csv',
    index=False, encoding='utf-8-sig'
)
print('\n✅ serie_mensual_2010_2024.csv guardada')

Forma: (3780, 18)
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Distritos: 21

Observaciones por distrito: 180 (esperado: 180)

covid_periodo marcado: 168 filas
(esperado: 8 meses × 21 distritos = 168)

=== ACC_PONDERADO MEDIO POR AÑO ===
año
2010    16.28
2011    17.71
2012    16.50
2013    17.51
2014    18.15
2015    16.46
2016    17.84
2017    16.73
2018    16.59
2019    14.22
2020    11.16
2021    11.57
2022    12.21
2023    13.07
2024    13.28

✅ serie_mensual_2010_2024.csv guardada


In [15]:
# CELDA 12 — Identificar meses COVID outliers

# Calculamos media mensual normal (excluyendo 2020)
# Usamos todos los años disponibles excepto 2020
media_normal = (
    tabla_mensual_completa[tabla_mensual_completa['año'] != 2020]
    .groupby('mes')['acc_ponderado']
    .mean()
    .round(2)
)

# Media mensual de 2020
media_2020 = (
    tabla_mensual_completa[tabla_mensual_completa['año'] == 2020]
    .groupby('mes')['acc_ponderado']
    .mean()
    .round(2)
)

# Comparación
comp = pd.DataFrame({
    'media_normal': media_normal,
    'media_2020':   media_2020
})
comp['diferencia_pct'] = (
    (comp['media_2020'] - comp['media_normal']) / comp['media_normal'] * 100
).round(1)

# Umbral: caída > 20% = outlier COVID
comp['outlier_covid'] = comp['diferencia_pct'] < -20

print('=== IMPACTO COVID POR MES (2020 vs media sin 2020) ===')
print(f'{"Mes":>5} {"Media normal":>15} {"Media 2020":>12} {"Diferencia %":>14} {"Outlier":>10}')
print('-' * 60)
for mes, row in comp.iterrows():
    outlier = '⚠️ Sí' if row['outlier_covid'] else '✅ No'
    print(f'{mes:>5} {row["media_normal"]:>15.2f} {row["media_2020"]:>12.2f} {row["diferencia_pct"]:>13.1f}% {outlier:>10}')

meses_outlier = comp[comp['outlier_covid']].index.tolist()
print(f'\nMeses a corregir: {meses_outlier}')

=== IMPACTO COVID POR MES (2020 vs media sin 2020) ===
  Mes    Media normal   Media 2020   Diferencia %    Outlier
------------------------------------------------------------
    1           14.29        15.62           9.3%       ✅ No
    2           14.46        11.62         -19.6%       ✅ No
    3           15.67         6.14         -60.8%      ⚠️ Sí
    4           14.76         1.81         -87.7%      ⚠️ Sí
    5           17.64        10.38         -41.2%      ⚠️ Sí
    6           17.66        13.24         -25.0%      ⚠️ Sí
    7           14.49        13.71          -5.4%       ✅ No
    8           10.13        10.24           1.1%       ✅ No
    9           16.03         9.57         -40.3%      ⚠️ Sí
   10           18.29        14.52         -20.6%      ⚠️ Sí
   11           17.30        13.05         -24.6%      ⚠️ Sí
   12           16.27        14.00         -14.0%       ✅ No

Meses a corregir: [3, 4, 5, 6, 9, 10, 11]


In [16]:
# CELDA 13 — Corregir meses COVID con interpolación

# Meses a corregir (caída > 15% respecto a media normal)
meses_covid = comp[comp['diferencia_pct'] < -15].index.tolist()
print(f'Meses a corregir: {meses_covid}')

serie_mensual_corregida = tabla_mensual_completa.copy()
serie_mensual_corregida['covid_imputado'] = 0

for distrito in serie_mensual_corregida['distrito'].unique():
    for mes in meses_covid:
        mask_2020 = (
            (serie_mensual_corregida['distrito'] == distrito) &
            (serie_mensual_corregida['año'] == 2020) &
            (serie_mensual_corregida['mes'] == mes)
        )

        # Valor del mismo mes en 2019 y 2021
        val_2019 = serie_mensual_corregida[
            (serie_mensual_corregida['distrito'] == distrito) &
            (serie_mensual_corregida['año'] == 2019) &
            (serie_mensual_corregida['mes'] == mes)
        ]['acc_ponderado'].values

        val_2021 = serie_mensual_corregida[
            (serie_mensual_corregida['distrito'] == distrito) &
            (serie_mensual_corregida['año'] == 2021) &
            (serie_mensual_corregida['mes'] == mes)
        ]['acc_ponderado'].values

        if len(val_2019) > 0 and len(val_2021) > 0:
            interpolado = round((val_2019[0] + val_2021[0]) / 2)
            serie_mensual_corregida.loc[mask_2020, 'acc_ponderado'] = interpolado
            serie_mensual_corregida.loc[mask_2020, 'covid_imputado'] = 1

# Verificación
print(f'\nFilas imputadas: {serie_mensual_corregida["covid_imputado"].sum()}')
print(f'Esperado: {len(meses_covid)} meses × 21 distritos = {len(meses_covid)*21}')

print('\n=== ACC_PONDERADO MEDIO POR MES 2020 — ANTES vs DESPUÉS ===')
antes  = tabla_mensual_completa[tabla_mensual_completa['año']==2020].groupby('mes')['acc_ponderado'].mean().round(2)
despues = serie_mensual_corregida[serie_mensual_corregida['año']==2020].groupby('mes')['acc_ponderado'].mean().round(2)
normal = media_normal

comp_final = pd.DataFrame({
    'normal': normal,
    'antes':  antes,
    'despues': despues
})
print(comp_final.to_string())

print('\n=== ACC_PONDERADO MEDIO POR AÑO (tras corrección) ===')
print(serie_mensual_corregida.groupby('año')['acc_ponderado'].mean().round(2).to_string())

# Guardamos
serie_mensual_corregida.to_csv(
    f'{outputs}/serie_mensual_2010_2024_corregida.csv',
    index=False, encoding='utf-8-sig'
)
print(f'\n✅ serie_mensual_2010_2024_corregida.csv guardada')
print(f'Forma: {serie_mensual_corregida.shape}')
print(f'Columnas: {serie_mensual_corregida.columns.tolist()}')

Meses a corregir: [2, 3, 4, 5, 6, 9, 10, 11]

Filas imputadas: 168
Esperado: 8 meses × 21 distritos = 168

=== ACC_PONDERADO MEDIO POR MES 2020 — ANTES vs DESPUÉS ===
     normal  antes  despues
mes                        
1     14.29  15.62    15.62
2     14.46  11.62    10.38
3     15.67   6.14    14.00
4     14.76   1.81    11.71
5     17.64  10.38    16.43
6     17.66  13.24    14.62
7     14.49  13.71    13.71
8     10.13  10.24    10.24
9     16.03   9.57    13.05
10    18.29  14.52    15.86
11    17.30  13.05    15.24
12    16.27  14.00    14.00

=== ACC_PONDERADO MEDIO POR AÑO (tras corrección) ===
año
2010    16.28
2011    17.71
2012    16.50
2013    17.51
2014    18.15
2015    16.46
2016    17.84
2017    16.73
2018    16.59
2019    14.22
2020    13.74
2021    11.57
2022    12.21
2023    13.07
2024    13.28

✅ serie_mensual_2010_2024_corregida.csv guardada
Forma: (3780, 19)
Columnas: ['distrito', 'año', 'mes', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mo

In [17]:
# ¿Qué umbral estadístico justificaría la corrección?

print('=== VARIABILIDAD NORMAL POR MES (excluyendo 2020) ===')
print(f'{"Mes":>5} {"Media":>10} {"Std":>8} {"CV%":>8} {"Umbral -1.5σ":>14} {"2020":>10} {"Corregir?":>12}')
print('-' * 75)

for mes in range(1, 13):
    valores = tabla_mensual_completa[
        (tabla_mensual_completa['mes'] == mes) &
        (tabla_mensual_completa['año'] != 2020)
    ]['acc_ponderado'].values

    media = valores.mean()
    std   = valores.std()
    cv    = (std / media * 100)
    umbral = media - 1.5 * std
    val_2020 = tabla_mensual_completa[
        (tabla_mensual_completa['mes'] == mes) &
        (tabla_mensual_completa['año'] == 2020)
    ]['acc_ponderado'].mean()

    corregir = '⚠️ Sí' if val_2020 < umbral else '✅ No'
    print(f'{mes:>5} {media:>10.2f} {std:>8.2f} {cv:>7.1f}% {umbral:>14.2f} {val_2020:>10.2f} {corregir:>12}')

=== VARIABILIDAD NORMAL POR MES (excluyendo 2020) ===
  Mes      Media      Std      CV%   Umbral -1.5σ       2020    Corregir?
---------------------------------------------------------------------------
    1      14.29    10.27    71.9%          -1.12      15.62         ✅ No
    2      14.46    10.30    71.2%          -0.99      11.62         ✅ No
    3      15.67    10.85    69.2%          -0.60       6.14         ✅ No
    4      14.76    10.17    68.9%          -0.49       1.81         ✅ No
    5      17.64    11.84    67.1%          -0.12      10.38         ✅ No
    6      17.66    10.50    59.5%           1.91      13.24         ✅ No
    7      14.49     9.85    68.0%          -0.28      13.71         ✅ No
    8      10.13     8.04    79.3%          -1.93      10.24         ✅ No
    9      16.03    10.64    66.4%           0.07       9.57         ✅ No
   10      18.29    11.39    62.3%           1.21      14.52         ✅ No
   11      17.30    10.98    63.5%           0.83      1